# Chapter 17 — GRPO from Scratch## Group Relative Policy Optimization[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**A100 GPU recommended. Runtime: ~45 minutes.**Implements GRPO (DeepSeek-R1's algorithm) from first principles and applies it to a mathematical reasoning task with verifiable rewards.

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport numpy as npimport matplotlib.pyplot as pltprint('Ready. For full training, an A100 GPU is recommended.')

## 17.1 GRPO: The Key InsightPPO needs a value function → expensive, can diverge.GRPO insight: for G responses to the same prompt, use the group as its own baseline:**A_i = (r_i - mean(r_1,...,r_G)) / std(r_1,...,r_G)**

In [ ]:
def grpo_advantages(rewards: list, eps: float = 1e-8) -> torch.Tensor:    """    Compute GRPO group-relative advantages.    rewards: list of G scalar rewards for the same prompt    """    r = torch.FloatTensor(rewards)    return (r - r.mean()) / (r.std() + eps)# Demonstrate with different reward scenariosscenarios = {    'All correct (easy problem)':      [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],    'Half correct (medium)':           [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0],    'Only one correct (hard)':         [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],    'All wrong (needs warm-up)':       [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],    'Partial credit distribution':     [0.0, 0.25, 0.25, 0.5, 0.5, 0.75, 1.0, 1.0],}fig, axes = plt.subplots(1, len(scenarios), figsize=(18, 4))for ax, (name, rewards) in zip(axes, scenarios.items()):    adv = grpo_advantages(rewards)    colors = ['seagreen' if a>0 else ('tomato' if a<0 else 'gray') for a in adv]    ax.bar(range(len(rewards)), adv.numpy(), color=colors)    ax.set_title(name, fontsize=9)    ax.axhline(0, color='black', linewidth=0.5)    ax.set_xlabel('Response index')    ax.set_ylabel('Advantage')plt.suptitle('GRPO: Group-Relative Advantages Under Different Reward Scenarios', fontsize=12)plt.tight_layout()plt.show()

## 17.2 Verifiable Math RewardMath answers are verified by SymPy — the objective, gameable-resistant reward signal behind DeepSeek-R1.

In [ ]:
try:    from sympy import sympify, simplify, Rational        def verify_math_answer(model_answer: str, correct_answer: str) -> float:        """        Verify a math answer using symbolic equality.        Returns 1.0 if correct, 0.0 otherwise.        """        try:            model_expr   = sympify(model_answer.replace('\\boxed{', '').replace('}', '').strip())            correct_expr = sympify(correct_answer)            if simplify(model_expr - correct_expr) == 0:                return 1.0            return 0.0        except:            return 0.0        # Test cases    print('Verifying math answers:')    test_cases = [        ('3',           '3',     1.0, 'Integer match'),        ('6/2',         '3',     1.0, 'Fraction simplified'),        ('1/3',         '0.333', 0.0, 'Not exactly equal'),        ('x**2 + 2*x',  'x*(x+2)', 1.0, 'Algebraic equivalence'),        ('wrong',       '3',     0.0, 'Wrong answer'),    ]    for model_ans, correct_ans, expected_reward, label in test_cases:        reward = verify_math_answer(model_ans, correct_ans)        status = '✓' if reward == expected_reward else '✗'        print(f'  {status} {label}: verify("{model_ans}", "{correct_ans}") = {reward}')except ImportError:    print('SymPy not installed. Run: pip install sympy')

## 17.3 GRPO Policy UpdateWith group-relative advantages computed, the GRPO loss mirrors PPO-Clip:

In [ ]:
def grpo_loss(log_probs_new, log_probs_old, advantages, epsilon=0.2, beta=0.04):    """    GRPO policy gradient loss.    log_probs_new: log π_θ(y|x)  [current policy, differentiable]    log_probs_old: log π_θ_old(y|x) [rollout policy, stopped gradient]    advantages: GRPO group-relative advantages    epsilon: PPO clip range    beta: KL penalty weight    """    ratio = torch.exp(log_probs_new - log_probs_old)    clipped_ratio = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)    policy_loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()    # KL penalty (approximated by log-ratio)    kl_penalty = (log_probs_new - log_probs_old).mean()    return policy_loss + beta * kl_penaltyprint('GRPO loss defined. Key differences from PPO:')print('  1. Advantages come from group comparison (not value function)')print('  2. No separate critic network or value loss')print('  3. KL penalty uses reference model, not old policy')

## ✅ Key Takeaways1. GRPO = PPO without the value function2. Group-relative advantages: A_i = (r_i - mean) / std across G responses per prompt3. The 'aha moment' emerges because extended reasoning gets rewarded by the verifier4. Cold start (SFT warm-up) is necessary when model accuracy < 5% on training tasks